# 第 27 节：全栈评测与最终交付 —— 让系统“可衡量、可证明、可交付”

> 前置课程：第 24 节（多智能体 Plan-and-Execute + ReAct 执行）、第 25 节（A2A 审核与反馈循环）、
>           第 26 节（RAG 知识库 + 长期 / 短期记忆）
> 前置知识：FastAPI / Vue3 基础；熟悉 SQLite“三张表”的演进故事（第 24-26 节）
> 对应项目：SmartBrief（`smartbrief/`）—— 本项目 **v27.0 完结版**
> 对应规格：`多智能体协作开发PRD（拆分）.md —— 子文档 6：第 27 节（全栈评测、API 完善与前端最终联调）`

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 课程导入：交付前的“灵魂三问” | — |
| **Part 2** | 环境准备与项目结构（新增 / 完善了哪些文件） | `backend/evaluation/*`、`db/*` 等 |
| **Part 3** | 先立框架：评测指标与用例集怎么设计 | `evaluation/metrics.py` |
| **Part 4** | 指标计算代码走读：口径即“宪法” | `evaluation/metrics.py` |
| **Part 5** | 批量执行：一场无人值守的“考试” | `evaluation/runner.py` |
| **Part 6** | 评测落库：一条评测记录的“一生” | `db/evaluation_repository.py` + `database.py` |
| **Part 7** | 后端 API：评测与报告怎么“开口说话” | `api/routes.py` |
| **Part 8** | 前端闭环：评测中心 / 报告中心 / 报告展示 | `frontend/src/views/*` |
| **Part 9** | 端到端演示（离线跑通评测链路，无需 API Key） | 多模块联合 |
| **Part 10** | 交付：一键启动与容器化部署 | `start.bat` / `start.sh` / `deploy/` |
| **Part 11** | 课程总结（项目完结） | — |

---

> **面向学员**：和前几节一样，本教程不是“背代码”，而是带你走一遍
> **真实项目迭代中遇到的问题 → 为什么这样设计 → 最终代码长什么样**。
> 这一节是系列的 **最后一节**：前几节让系统“会对话、会用工具、会分工、会把关、
> 有知识、有记忆”—— 本节再补上两块拼图：**① 一套自动化“考试系统”（评测）
> ② 一条能交付给别人跑的链路（一键启动 + 容器 + 云服务器部署）**。
> 建议按顺序逐 Cell 运行，观察每一次输出。

---
## Part 1：课程导入 —— 交付前的“灵魂三问”
---

### 1.1 产品经理又来了

你交付了第 26 节的“有知识、有记忆”版本。功能看起来很全，但产品经理在**验收与交付**前
向你抛出三个直击灵魂的问题：

> **问题一（没有“考卷”）**：“你说系统现在很厉害了，**拿什么证明**？每次演示都是你现场敲一个
> 主题、跑通了再给我看。如果它平均 10 次里挂 3 次、写报告要 20 分钟、同主题跑两遍结构都不一样，
> 你拿什么数字说服我上线？”

> **问题二（报告没有出口）**：“我做了 20 个主题的报告，现在要按主题翻历史报告，
> 只能去任务列表一条条点开详情找 —— **有没有一个‘报告中心’，能搜、能排序、能导出？**”

> **问题三（交付 ≠ 能跑起来）**：“你自己电脑上跑得再欢，客户那边怎么用？总不能让我
> 把命令行教程发过去，让客户自己 `pip install` + 配 Key 吧？有没有**一键启动**，
> 甚至一条命令**部署到云服务器**的方案？”

把三个问题放在一起看，其实是同一个大主题：

> **系统只完成了“功能开发”，还没完成“工程化交付”** —— 不能量化验收、
> 报告无统一出口、本地能跑不等于可交付运行。

> 🤔 **暂停思考**：如果让你来回答这三个问题，第一版方案会是什么？
> 手动挑几个主题跑一遍、拿记事本记结果？把报告导出成文件放到某个目录？
> 写一份“部署手册”让客户照着敲？先把你的直觉方案写下来 —— 后面你会看到，
> 这三个“直觉方案”为什么都行不通。

---
### 1.2 拆解：真实团队怎么验收一个 Agent 系统？

把三个问题拆开，对应到真实研发团队的三个“岗位动作”：

| 你的“直觉方案” | 问题 | 工程上对应什么 |
|---------------|------|---------------|
| 手动挑主题跑一遍、拿记事本记 | 不可复现：下次演示又得重跑；标准不统一：换个人验收口径就变 | **自动化评测**：固定用例集 + 固定指标口径，一键批量执行 |
| 报告存成散文件 / 靠人肉翻详情页 | 散落无索引：主题、时间、状态没有统一查询入口 | **报告中心**：`tasks` 表 + 搜索 / 排序 / 删除 / 展示 API |
| 写一份“安装手册”让别人照做 | 环境差异、依赖冲突、命令太多，演示时最容易翻车 | **一键启动脚本 + 容器化**：双击即跑、镜像即交付 |
| 上线前不知道“这次改坏了没” | 没有回归基准，改一行代码没人敢保证 | **评测报告留档**：历史评测可对比，指标从绿变红立刻暴露 |

> ⚠️ 一个关键认知：**“能跑通一次”和“能证明稳定可用”是两回事**。
> 多智能体系统每一步都调 LLM，是**非确定性的** —— 同一个主题跑两次结果都可能不一样。
> 越是这样的系统，越需要“自动化、可复现、口径固定”的验收手段。
> 这正是第 27 节 `evaluation/`（评测引擎）存在的意义：**让系统自己给自己打分**。

### 1.3 本节课的“开发旅程”

```
痛点 → 试错 → 最终设计 → 项目里的最终代码 → 端到端演示
```

| 步骤 | 我们解决什么 | 项目里的实现 |
|------|-------------|-------------|
| ① | 没有“考卷”：系统好不好无法量化 | 10 个标准用例 + 五维指标体系（Part 3 / 4） |
| ② | 手动验收不可复现、口径不一致 | 纯计算指标模块 + 固定 TARGETS + 达标判定（Part 4） |
| ③ | 批量跑完没进度、不知道挂没挂 | `runner` 后台执行 + `_running` 并发锁 + 逐用例进度落库（Part 5） |
| ④ | 评测结果没有归宿、重启就丢 | `evaluation_reports` 表：create → progress → finish / fail（Part 6） |
| ⑤ | 报告散落、无法检索与展示 | 评测 / 报告 / 列表搜索排序 API（Part 7） |
| ⑥ | 数字不直观、客户看不懂 | 前端评测中心（进度条 + 指标卡 + 雷达图）（Part 8） |
| ⑦ | 本地能跑 ≠ 可交付 | 一键启动 + Docker Compose + 云服务器部署脚本（Part 10） |

---
## Part 2：环境准备与项目结构
---

### 2.1 第 27 节动了哪些文件？—— 先有个全景

第 27 节**没有新增后端依赖**（评测完全复用已有的 LangGraph 工作流 + SQLite），
但新增了一个全新的后端包 `evaluation/`，并“完善”了一批既有文件。先记住这张清单：

**后端（新增）**

| 文件 | 作用 |
|------|------|
| `backend/evaluation/metrics.py` | ★ 五维指标**纯计算**模块（成功率 / 耗时 / P95 / ReAct 循环 / 关键词覆盖率 / 记忆命中率） |
| `backend/evaluation/runner.py` | ★ 评测引擎：10 个标准用例 + 批量执行 + 并发锁 + 进度 |

**后端（完善）**

| 文件 | 第 27 节做了什么 |
|------|-----------------|
| `backend/db/database.py` | 新增 `evaluation_reports` 表（与 tasks 同库同文件） |
| `backend/db/evaluation_repository.py` | ★ 评测报告 CRUD（create / set_progress / finish / fail / fail_running_interrupted / get / list） |
| `backend/api/routes.py` | 评测三 API + 删除 + `GET /tasks/{id}/report` + `GET /tasks` 扩展 `keyword` / `order` |
| `backend/main.py` | lifespan 启动兜底：遗留 `running` 评测 → `failed`（与任务侧 `paused` 对称） |

**前端（新增 / 完善）**

| 文件 | 作用 |
|------|------|
| `frontend/src/views/Evaluation.vue` | ★ 评测中心页（路由 `/evaluation`） |
| `frontend/src/views/Reports.vue` | ★ 报告中心页（路由 `/reports`） |
| `frontend/src/views/ReportView.vue` | ★ 报告展示页（路由 `/report/:taskId`） |
| `frontend/src/api/evaluation.ts` | ★ 评测 API 封装（类型 + 运行 / 列表 / 详情 / 删除） |
| `frontend/src/api/tasks.ts` | 扩展 `fetchTaskReport` + `fetchTasks` 的 `keyword` / `order` 参数 |
| `frontend/package.json` | 新增依赖 **echarts**（雷达图） |

**测试与交付（新增 / 完善）**

| 文件 | 作用 |
|------|------|
| `backend/tests/test_evaluation.py` / `test_evaluation_api.py` | 指标 / 仓储 / runner / API 四组测试 |
| `start.bat` / `start.sh` / `deploy/` | 一键启动与容器化部署（Part 10） |

> 读代码前先记住一句话：**第 27 节所有改动，都是在为“交付”服务 ——**
> **① 先定口径（指标）② 再造考卷（用例）③ 再能留档（落库）④ 再能看见（API + 前端）⑤ 最后能交付（部署）**。

In [1]:
# ============================================================
# Cell 1：检查环境 + 把 backend 加入 sys.path
# ============================================================
import os
import sys

print("=" * 55)
print("  环境检查（第 27 节）")
print("=" * 55)
print(f"  Python 版本: {sys.version.split()[0]}")

# 兼容两种运行位置：
#   1) notebook 在项目根目录运行时 -> backend = ./smartbrief/backend
#   2) notebook 在 smartbrief 目录运行时 -> backend = ./backend
BACKEND = os.path.join(os.getcwd(), "smartbrief", "backend")
if not os.path.isdir(BACKEND):
    BACKEND = os.path.join(os.getcwd(), "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)
FRONTEND = os.path.join(os.path.dirname(BACKEND), "frontend")
print(f"  backend 路径: {BACKEND}")
print(f"  frontend 路径: {FRONTEND}")

for name in ("langgraph", "chromadb"):
    try:
        __import__(name, fromlist=["*"])
        print(f"  [OK] {name} 已安装")
    except Exception as exc:
        print(f"  [警告] 缺少 {name}: {exc}")

# 第 27 节核心包可导入性检查（真正跑评测的是 evaluation.runner）
try:
    from evaluation import metrics, runner  # noqa
    from db import evaluation_repository  # noqa
    print("  [OK] evaluation/metrics + runner 与 db/evaluation_repository 可导入")
    print(f"  [OK] 当前启用的标准用例数: runner.case_count() = {runner.case_count()}")
except Exception as exc:
    print("  [警告] 模块导入失败（请确认当前目录在项目根或 smartbrief 下）:", exc)


  环境检查（第 27 节）
  Python 版本: 3.10.20
  backend 路径: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj06\smartbrief\backend
  frontend 路径: c:\Users\13691\Documents\Code\260601-GPKJ\05_multi-agent\proj06\smartbrief\frontend
  [OK] langgraph 已安装
  [OK] chromadb 已安装
  [OK] evaluation/metrics + runner 与 db/evaluation_repository 可导入
  [OK] 当前启用的标准用例数: runner.case_count() = 1


In [2]:
# ============================================================
# Cell 2：文件探测 —— 第 27 节新增 / 完善的文件真实存在吗？
# ============================================================
import json
from pathlib import Path

print("=" * 55)
print("  后端 evaluation/ 包结构")
print("=" * 55)
for p in sorted((Path(BACKEND) / "evaluation").rglob("*.py")):
    print("    " + str(p.relative_to(Path(BACKEND))))

print("=" * 55)
print("  第 27 节关键文件探测（含前端）")
print("=" * 55)
checks = [
    ("db/evaluation_repository.py（评测 CRUD）", Path(BACKEND) / "db" / "evaluation_repository.py"),
    ("api/routes.py（评测 + 报告 API）", Path(BACKEND) / "api" / "routes.py"),
    ("main.py（lifespan 兜底）", Path(BACKEND) / "main.py"),
    ("tests/test_evaluation.py（测试）", Path(BACKEND) / "tests" / "test_evaluation.py"),
    ("tests/test_evaluation_api.py（测试）", Path(BACKEND) / "tests" / "test_evaluation_api.py"),
    ("views/Evaluation.vue（评测中心）", Path(FRONTEND) / "src" / "views" / "Evaluation.vue"),
    ("views/Reports.vue（报告中心）", Path(FRONTEND) / "src" / "views" / "Reports.vue"),
    ("views/ReportView.vue（报告展示）", Path(FRONTEND) / "src" / "views" / "ReportView.vue"),
    ("api/evaluation.ts（前端评测 API）", Path(FRONTEND) / "src" / "api" / "evaluation.ts"),
]
for label, p in checks:
    print(f"  [{'有' if p.exists() else '缺'}] {label}")

print()
print("  package.json 中的图表 / 渲染相关依赖：")
try:
    pkg = json.load(open(Path(FRONTEND) / "package.json", encoding="utf-8"))
    all_deps = {**pkg.get("dependencies", {}), **pkg.get("devDependencies", {})}
    for dep in ("echarts", "marked", "dompurify", "html2pdf.js"):
        if dep in all_deps:
            print(f"    {dep}: {all_deps[dep]}")
except Exception as exc:
    print("    [读取失败]", exc)


  后端 evaluation/ 包结构
    evaluation\__init__.py
    evaluation\metrics.py
    evaluation\runner.py
  第 27 节关键文件探测（含前端）
  [有] db/evaluation_repository.py（评测 CRUD）
  [有] api/routes.py（评测 + 报告 API）
  [有] main.py（lifespan 兜底）
  [有] tests/test_evaluation.py（测试）
  [有] tests/test_evaluation_api.py（测试）
  [有] views/Evaluation.vue（评测中心）
  [有] views/Reports.vue（报告中心）
  [有] views/ReportView.vue（报告展示）
  [有] api/evaluation.ts（前端评测 API）

  package.json 中的图表 / 渲染相关依赖：
    echarts: ^5.6.0
    marked: ^18.0.11
    dompurify: ^3.4.14
    html2pdf.js: ^0.14.0


### 2.2 项目结构一览（第 27 节新增 / 完善部分）

```
smartbrief/
├── backend/
│   ├── evaluation/                  # ★ 新增整个包：自动化评测引擎
│   │   ├── metrics.py               #    五维指标纯计算 + TARGETS 目标值 + Markdown 报告
│   │   └── runner.py                #    10 个标准用例（当前启用 1 个）+ 批量执行 + 并发锁
│   ├── db/
│   │   ├── database.py              # ★ 新增 evaluation_reports 表（建表 SQL 幂等）
│   │   └── evaluation_repository.py # ★ 新增：评测报告 CRUD（JSON 字段序列化封装）
│   ├── api/routes.py                # ★ 评测三 API + 删除 + 任务报告 + 列表 keyword/order
│   ├── main.py                      # ★ lifespan：遗留 running 评测 → failed
│   └── tests/test_evaluation*.py    # ★ 指标 / 仓储 / runner / API 测试
├── frontend/src/
│   ├── api/evaluation.ts            # ★ 评测 API 封装（类型 + 4 个函数）
│   └── views/
│       ├── Evaluation.vue           # ★ 评测中心：运行 + 进度条 + 指标卡 + 雷达图 + 明细
│       ├── Reports.vue              # ★ 报告中心：已生成报告列表 + 搜索/排序/删除
│       └── ReportView.vue           # ★ 报告展示：Markdown + 执行摘要 + 导出
├── start.bat / start.sh             # ★ 一键启动（Part 10）
└── deploy/                          # ★ 容器化部署：docker-compose + 云服务器脚本（Part 10）
```

> 注意：**评测模块不长在 agents/ 目录里，而是独立的 `evaluation/` 包** ——
> 因为它读的是“任务的产物”（tasks / task_logs 表），而不是某个 Agent 的实现。
> **面向产物的“考官”要和面向过程的“考生”（agents/）解耦**，将来换成别的智能体系统，
> 这套评测引擎照样能用。

---
## Part 3：先立框架 —— 评测指标与用例集怎么设计
---

动手写代码前，先回答两个“设计题”：**考什么**（指标）与 **考哪些题**（用例集）。

### 3.1 指标设计：验收标准必须能写成数字

产品经理的问题一（“拿什么证明”）翻译成工程语言就是：**定义一组可自动计算的验收指标**。
本项目对应 PRD「模块六」的轻量化五维指标：

| 指标 | 计算方式 | 目标值 | 它回答什么 |
|------|---------|--------|-----------|
| `success_rate` 任务成功率 | 成功任务数 / 总任务数 | ≥ 90% | 系统**靠不靠谱**（会不会动不动挂） |
| `avg_duration_seconds` 平均耗时 | 成功任务总耗时 / 成功任务数 | ≤ 120s | 系统**快不快**（协同效率） |
| `avg_react_loops` 平均 ReAct 循环 | 总 Action 日志条数 / 任务数 | ≤ 8 次 | 系统**绕不绕**（思考→行动→观察是否收敛） |
| `keyword_coverage_rate` 关键词覆盖率 | 命中关键词数 / 总关键词数（平均） | ≥ 70% | 系统**写没写全**（报告覆盖用户关心的要点） |
| `memory_hit_rate` 记忆命中率 | 有 `memory_hits` 的任务数 / 总任务数 | ≥ 30% | 系统**用没用记忆**（第 26 节能力是否真被调用） |

> 🎯 三个设计取向（为什么是“这五个”而不是别的）：
>
> 1. **必须能“自动”算**：所有指标都从 `tasks` / `task_logs` / `memory_hits` 等**已落库的结构化字段**
>    统计而来，不需要人去看报告内容 —— 这是“一键评测”的前提。
> 2. **关键词覆盖率用“子串匹配”，不请 LLM 打分**：LLM 打分更“聪明”但贵、慢、且每次结果可能不同；
>    子串匹配便宜、确定、可复现 —— **评测口径宁要稳定的“傻办法”，不要聪明的“玄学”**。
> 3. **目标值写死在 `TARGETS` 一个常量里**：改口径只改一处；前端指标卡、雷达图的达标判定全部读它。

### 3.2 用例集设计：10 个标准 Topic 为什么长这样？

有了“尺子”，还得有“考卷” —— `runner.TEST_CASES`。设计上有两个刻意的选择：

| 设计点 | 为什么 |
|--------|--------|
| **固定 10 个行业 Topic**（储能 ×2 / 消费电子 ×3 / 智能家居 / 出行 / 新能源 / 工业 / 个护） | 覆盖足够广的领域，避免“只在某一类主题上表现好”的幸存者偏差 |
| **主题刻意对齐 `data/templates/` 的预置模板行业** | 让第 26 节的 Planner 模板 Few-shot 与 Writer RAG 术语注入**更可能命中** —— 评测要能观察到“知识 / 记忆有没有生效”，而不是白测 |
| **每个 Topic 附 `keywords` 预期关键词** | 这就是该题的“标准答案”，用于算关键词覆盖率 |
| **教学取舍：当前仅启用第 1 个用例，其余 9 个注释** | 10 个用例真实跑一遍要 5-10 分钟；教学演示先只跑 1 个快速验证链路。**恢复 = 取消注释**，`case_count()` 用 `len(TEST_CASES)` 动态取值 —— 跑几个就报几个，绝不硬编码 |

Cell 3 直接看常量与用例：

In [3]:
# ============================================================
# Cell 3：先看“宪法” —— 指标目标值与标准用例集
# ============================================================
from evaluation import metrics, runner

print("=" * 55)
print("  metrics.TARGETS（目标值 = 达标判定基准，前端也读它）")
print("=" * 55)
for k, v in metrics.TARGETS.items():
    print(f"    {k:<24} = {v}")

print()
print("=" * 55)
print("  metrics.TARGET_LABELS（前端指标卡文案）")
print("=" * 55)
for k, v in metrics.TARGET_LABELS.items():
    print(f"    {k:<24} = {v}")

print()
print("=" * 55)
print("  runner.TEST_CASES（标准用例集，当前启用 1 / 10）")
print("=" * 55)
for i, case in enumerate(runner.TEST_CASES, start=1):
    print(f"    [{i}] topic    = {case['topic']}")
    print(f"         keywords = {case['keywords']}")

print()
print(f"  case_count() = {runner.case_count()}  （动态 len(TEST_CASES)，启用几个就报几个）")
print()
print("  说明：本 Notebook 的端到端演示（Part 9）会自动用这 1 个启用的用例，")
print("        想体验完整 10 用例：取消 runner.py 里其余 9 条的注释即可。")


  metrics.TARGETS（目标值 = 达标判定基准，前端也读它）
    success_rate             = 0.9
    avg_duration_seconds     = 120.0
    avg_react_loops          = 8.0
    keyword_coverage_rate    = 0.7
    memory_hit_rate          = 0.3

  metrics.TARGET_LABELS（前端指标卡文案）
    success_rate             = 任务成功率
    avg_duration_seconds     = 平均执行耗时（秒）
    avg_react_loops          = 平均 ReAct 循环次数
    keyword_coverage_rate    = 关键词覆盖率
    memory_hit_rate          = 记忆命中率（长期记忆被引用）

  runner.TEST_CASES（标准用例集，当前启用 1 / 10）
    [1] topic    = 便携式储能电源 2026 北美市场
         keywords = ['市场规模', '增长率', '头部玩家', '用户痛点', '便携储能']

  case_count() = 1  （动态 len(TEST_CASES)，启用几个就报几个）

  说明：本 Notebook 的端到端演示（Part 9）会自动用这 1 个启用的用例，
        想体验完整 10 用例：取消 runner.py 里其余 9 条的注释即可。


---
## Part 4：指标计算代码走读 —— 口径即“宪法”
---

### 4.1 设计取向：为什么 metrics.py 是“纯计算、零依赖”？

打开 `metrics.py` 前，先看文件头注释里的承诺：

> 本模块负责**纯计算（无 IO、无依赖），便于单测**；
> runner 负责批量执行并收集每个用例的原始数据，最终调用本模块汇总。

这条约定把评测拆成了**两半**：

```
runner（做：跑任务、计时、采样）──原始数据──► metrics（做：汇总、判达标、出报告）
     │                                            │
     └── 有 IO、有网络、有异常                     └── 纯函数：给什么算什么，不会失败
```

> 为什么这样切？—— 指标计算是最需要**被信任**的部分（它是验收依据），
> 而“信任”只能来自测试。纯函数没有文件 / 网络依赖，几十行单测就能把口径钉死；
> 如果把它和批量执行混在一起，每次测试都要 mock 一大堆 IO，成本高且容易漏测。

### 4.2 四个“口径陷阱”（读代码时重点体会）

| 陷阱 | naive 写法 | 项目的口径 | 为什么 |
|------|-----------|-----------|--------|
| 平均耗时该不该算失败任务？ | `总耗时 / 总任务数` | **只算成功任务**（失败任务的 `duration_seconds` 为 `None`，不计） | 失败任务可能 0.01s 就抛异常，拉低平均值造成“假快”；PRD 明确：耗时只看成功任务 |
| P95 样本太少怎么办？ | 硬算百分位（插值出荒唐值） | **不足 20 个样本退化为最大值** | 小样本插值没有统计意义，取最大值至少保守、真实 |
| ReAct“循环次数”怎么数？ | 数所有日志条数 | **只数 `log_type == 'Action'` 的条数** | 一次循环 = 一次思考→行动→观察，Action 是循环的“心跳” |
| 关键词“命中”怎么判定？ | 精确匹配 | **子串匹配 + 大小写不敏感** | 报告里关键词常被包在句子里（“…市场规模约…”），子串命中更贴近真实覆盖 |

Cell 4 / Cell 5 走读最终代码：

In [4]:
# ============================================================
# Cell 4：走读 metrics.py —— 三个“口径敏感”的小函数
# ============================================================
import inspect
from evaluation import metrics

for fn_name in ("compute_keyword_coverage", "count_react_actions", "compute_p95"):
    fn = getattr(metrics, fn_name)
    print("=" * 55)
    print(f"  metrics.{fn_name}")
    print("=" * 55)
    print(inspect.getsource(fn))


  metrics.compute_keyword_coverage
def compute_keyword_coverage(text: str, keywords: Iterable[str]) -> float:
    """单个用例的关键词覆盖率：命中关键词数 / 总关键词数（0~1）。

    教学要点：这是最朴素的“关键词命中”——按子串匹配报告全文
    （大小写不敏感），用于衡量报告是否覆盖了用户关心的要点。
    """
    keywords = [k for k in keywords if k]
    if not keywords:
        return 0.0
    lowered = (text or "").lower()
    hit = sum(1 for k in keywords if k.lower() in lowered)
    return round(hit / len(keywords), 4)

  metrics.count_react_actions
def count_react_actions(logs: Iterable[dict]) -> int:
    """统计 ReAct 循环次数：= 该任务的全部 Action 类日志条数。

    教学要点：每次 Action 代表一次“思考 → 行动 → 观察”循环，
    task_logs 表里 log_type='Action' 的条数即循环次数（评测用）。
    """
    return sum(1 for log in logs or [] if log.get("log_type") == "Action")

  metrics.compute_p95
def compute_p95(values: Iterable[float]) -> float:
    """P95 耗时：排序后取第 95 百分位（不足 20 个样本时放宽取最大）。

    教学要点：P95 比平均值更能反映“尾部慢任务”体验；
    小样本下退化为最大值，避免空洞的插值结果。
    """
    values = sorted(float(v) for v in values)
    if not values:
  

In [5]:
# ============================================================
# Cell 5：走读 metrics.py —— 汇总 compute_metrics（达标判定核心）
# ============================================================
import inspect
from evaluation import metrics

print("=" * 55)
print("  metrics.compute_metrics（五维汇总 + met 达标判定）")
print("=" * 55)
print(inspect.getsource(metrics.compute_metrics))


  metrics.compute_metrics（五维汇总 + met 达标判定）
def compute_metrics(results: list[dict]) -> dict:
    """汇总全部用例结果 → 5 项指标实测值 + 达标判定。

    :param results: runner 收集的用例结果列表，每个元素含：
        success(bool) / duration_seconds(float) / react_loops(int) /
        keyword_coverage(float) / memory_hits(int) / topic(str)
    :return: {summary, targets, met} —— summary 为实测值汇总，
        targets 为目标值，met 为逐项是否达标（前端雷达图与状态标签数据源）
    """
    total = len(results)
    success = [r for r in results if r.get("success")]
    success_rate = _rate(len(success), total)
    # 平均耗时按“成功任务”口径（失败的用例耗时无意义，PRD：总耗时/成功任务数）
    durations = [r["duration_seconds"] for r in success if r.get("duration_seconds") is not None]
    avg_duration = _avg(durations)
    p95_duration = compute_p95(durations)
    loops = _avg([r.get("react_loops", 0) for r in results])
    coverage = _avg([r.get("keyword_coverage", 0.0) for r in results])
    memory_hit_rate = _rate(sum(1 for r in results if r.get("memory_hits", 0) > 0), total)
    summary 

### 4.3 达标判定 `met`：评测报告的“红绿灯”

`compute_metrics` 的返回值是三层结构，先记住它的形状（前端整个评测中心页面都是它的消费者）：

```python
{
  "summary": {   # 实测值汇总
     "total_cases": 10, "success_cases": 9, "failed_cases": 1,
     "success_rate": 0.9, "avg_duration_seconds": 66.0, ...
  },
  "targets":  {  # 目标值（原样引用 TARGETS）
     "success_rate": 0.90, "avg_duration_seconds": 120.0, ...
  },
  "met":      {  # 逐项是否达标（前端 ✅/❌ 标签 + 雷达图数据源）
     "success_rate": True, "keyword_coverage_rate": True, ...
  }
}
```

> 注意达标方向不一样：成功率 / 覆盖率 / 记忆命中率是 **≥ 目标**，耗时 / ReAct 次数是 **≤ 目标**。
> 到 Part 8 会看到前端雷达图为了统一“越高越好”，把耗时 / 循环做了 `1 - 实测/目标` 的归一化。

Cell 6 用三份“假结果”手工体验一次完整计算 + Markdown 报告（纯函数，不联网）：

In [6]:
# ============================================================
# Cell 6：用手工“假结果”体会 compute_metrics + build_markdown
# ============================================================
from evaluation import metrics

# 三份“假结果”：2 成功 + 1 失败；成功用例的关键词覆盖率一高一低
fake_results = [
    {"case_index": 1, "topic": "便携式储能电源 2026 北美市场",
     "success": True, "duration_seconds": 55.0, "react_loops": 5,
     "keyword_coverage": 1.0, "memory_hits": 2},
    {"case_index": 2, "topic": "智能手表 2026 中国市场",
     "success": True, "duration_seconds": 320.0, "react_loops": 9,
     "keyword_coverage": 0.4, "memory_hits": 0},
    {"case_index": 3, "topic": "工业无人机 2026 市场调研",
     "success": False, "duration_seconds": None, "react_loops": 0,
     "keyword_coverage": 0.0, "memory_hits": 0},
]

data = metrics.compute_metrics(fake_results)
print("=" * 55)
print("  summary（实测值）")
print("=" * 55)
for k, v in data["summary"].items():
    print(f"    {k:<24} = {v}")

print()
print("=" * 55)
print("  met（达标判定 —— 方向不同：成功率≥目标，耗时/循环≤目标）")
print("=" * 55)
for k, ok in data["met"].items():
    print(f"    {k:<24} = {'✅ 达标' if ok else '❌ 未达标'}")

print()
print("=" * 55)
print("  build_markdown 生成的 Markdown 报告（前端评测中心直接渲染它）")
print("=" * 55)
print(metrics.build_markdown("eval_demo_metrics", fake_results, data))


  summary（实测值）
    total_cases              = 3
    success_cases            = 2
    failed_cases             = 1
    success_rate             = 0.6667
    avg_duration_seconds     = 187.5
    p95_duration_seconds     = 320.0
    avg_react_loops          = 4.67
    keyword_coverage_rate    = 0.47
    memory_hit_rate          = 0.3333

  met（达标判定 —— 方向不同：成功率≥目标，耗时/循环≤目标）
    success_rate             = ❌ 未达标
    avg_duration_seconds     = ❌ 未达标
    avg_react_loops          = ✅ 达标
    keyword_coverage_rate    = ❌ 未达标
    memory_hit_rate          = ✅ 达标

  build_markdown 生成的 Markdown 报告（前端评测中心直接渲染它）
# SmartBrief 自动化评测报告

- 评测 ID：`eval_demo_metrics`
- 生成时间：2026-09-03 13:00:22
- 用例数：3（成功 2 / 失败 1）

## 一、指标总览

| 维度 | 指标 | 实测值 | 目标值 | 达标 |
| --- | --- | --- | --- | --- |
| 任务完成 | 任务成功率 | 66.7% | 90.0% | ❌ |
| 任务完成 | 平均执行耗时（秒） | 187.5s | 120.0s | ❌ |
| 任务完成 | 平均 ReAct 循环次数 | 4.7 次 | 8.0 次 | ✅ |
| 任务完成 | 关键词覆盖率 | 47.0% | 70.0% | ❌ |
| 任务完成 | 记忆命中率（长期记忆被引用） | 33.3% | 30.0% | ✅ |

## 二、用例明细

| # |

---
## Part 5：批量执行 —— 一场无人值守的“考试”
---

### 5.1 批量执行怎么设计？三个选择背后的理由

`runner.py` 负责“发卷子、收卷子、登记分数”。看代码前先过三个设计决策：

**决策一：进程内直接调 `workflow.run_task`，而不是循环请求 `POST /generate`**

```
PRD 早期设想：for topic in TEST_CASES:  requests.post("http://localhost:8002/api/v1/generate")
最终实现：    for case in TEST_CASES:    await workflow.run_task(task_id, topic)   # 进程内直调
```

| 方案 | 优点 | 缺点 |
|------|------|------|
| HTTP 回环自调用 | 最接近真实用户链路 | 要自建服务 / 处理端口占用与网络抖动，慢且脆弱 |
| 进程内直调 | 快、稳、可 mock（测试友好） | 少了 HTTP 一层 —— 但对评测无影响：测的是“工作流”，不是“路由” |

> 教学要点：`_run_single_case` 的 docstring 写得很清楚 —— 它与 `POST /generate` 的提交逻辑
> **完全一致**（先落库 running 初始记录，再异步跑图），只是多了**计时与采样**。

**决策二：计时用 `time.monotonic()`，不用 `time.time()`**

`time.time()` 返回“墙上时钟”，会被**系统改时间 / NTP 校时**影响（往前拨一小时，耗时变负数）；
`time.monotonic()` 是**单调时钟**，只增不减，专为“测间隔”设计 —— 评测计时必须用它。

**决策三：结果里为什么带 `forced_pass` 字段？**

每个用例的结果不但记 `success`，还记 `forced_pass`（第 25 节 A2A 审核是否“强制放行”）。
这样评测能回答更细的问题：**“任务成功”是高质量通过，还是“超限被强行放行”？**
—— 考官连“放水”都要记录在案。

### 5.2 单个用例失败 ≠ 整轮评测失败（重要口径）

| 层级 | 异常 | 处理 |
|------|------|------|
| **单用例**（`_run_single_case`） | 某用例跑挂（如 LLM 超时） | `except` 捕获 → 记为 `success=False` 并附 `error` → **继续跑下一个** |
| **整轮**（`_execute`） | 整体层异常（如进度落库失败） | 外层 `except` → `evaluation_repository.fail()` → 整轮状态 `failed` |

> 一句话：**用例可以挂，考试不能崩**。成功率的“分母”始终是总用例数，
> 挂掉的用例也进分母 —— 这正是成功率指标的意义。

Cell 7 走读 `_run_single_case`：

In [7]:
# ============================================================
# Cell 7：走读 runner.py —— 单个用例怎么“考”
# ============================================================
import inspect
from evaluation import runner

print("=" * 55)
print("  runner._run_single_case（计时 + 跑图 + 采样五个指标）")
print("=" * 55)
print(inspect.getsource(runner._run_single_case))


  runner._run_single_case（计时 + 跑图 + 采样五个指标）
async def _run_single_case(case: dict, case_index: int) -> dict:
    """执行单个用例：创建任务记录 → 跑完整工作流 → 收集原始数据。

    教学要点：与 `POST /api/v1/generate` 的提交逻辑完全一致
    （先落库 running 初始记录，再异步跑图），仅多了计时与采样。
    用例结果逐项收集：耗时 / 成功与否 / ReAct 循环次数（Action 日志数）/
    关键词覆盖率（报告全文子串匹配）/ 记忆命中数（memory_hits 长度）。
    """
    task_id = f"task_{uuid4().hex[:8]}"
    task_repository.create_task(task_id, case["topic"])
    started = time.monotonic()
    try:
        await workflow.run_task(task_id, case["topic"])
        duration = time.monotonic() - started
        task = task_repository.get_task(task_id)
        logs = task_repository.get_logs(task_id)
        report_text = (task or {}).get("final_report") or ""
        return {
            "case_index": case_index,
            "topic": case["topic"],
            "keywords": case["keywords"],
            "task_id": task_id,
            "success": True,
            "duration_seconds": round(duration, 2),
            "react_lo

### 5.3 `start_evaluation`：为什么“立即返回 eval_id”而不是同步等结果？

10 个用例真实跑要 **5-10 分钟**，HTTP 接口不可能让用户干等。所以触发接口的语义是：

```
POST /evaluation/run
   │
   ├─ 检查并发锁 _running（已在跑 → RuntimeError → API 400）
   ├─ _running = True
   ├─ 生成 eval_id（eval_xxxx），先落库 status=running   ← 前端立刻能查到“开始了一条评测”
   └─ asyncio.create_task(_execute(eval_id))            ← 后台慢慢跑，接口立即返回
       返回 {"eval_id": ..., "status": "running", "total_cases": ...}
```

> 这个“**先落 running 记录 + 后台任务**”的模式和第 24 节 `POST /generate` 提交任务
> **一模一样** —— 长耗时的活儿，永远别让接口傻等，而是给客户端一个 id 去**轮询进度**。

**并发锁 `_running`**：评测要批量调 LLM（费钱费时），同一时刻只允许一场 ——
重复触发直接抛 `RuntimeError`，API 层转成 **400**，前端按钮提示“已有评测在运行”。

### 5.4 中断兜底：进程重启后，那个“跑到一半”的评测怎么办？

评测在 `asyncio.create_task` 后台执行，如果进程被强杀，它**没机会写终态**，
数据库会留一条永远 `running` 的记录 —— 前端评测中心进度条卡死、「运行评测」按钮被禁用。

所以 `main.py` 的 lifespan 在启动时做一次“归位”（与任务侧对称）：

| 记录类型 | 谁兜底 | 归位成什么 | 为什么不同 |
|---------|--------|-----------|-----------|
| 遗留 `running` 任务（第 26 节） | `mark_interrupted_tasks_as_paused` | `paused`（可恢复执行） | 任务有 Checkpointer 快照，可断点续跑 |
| 遗留 `running` 评测（第 27 节） | `fail_running_interrupted` | `failed`（可重新触发） | 评测是“重跑型”批量任务，没有续跑价值，标 failed 让用户重开即可 |

Cell 8 走读 `start_evaluation` + `_execute`：

In [8]:
# ============================================================
# Cell 8：走读 runner.py —— 触发入口 + 后台整轮执行
# ============================================================
import inspect
from evaluation import runner

for fn_name in ("start_evaluation", "_execute"):
    print("=" * 55)
    print(f"  runner.{fn_name}")
    print("=" * 55)
    print(inspect.getsource(getattr(runner, fn_name)))


  runner.start_evaluation
def start_evaluation() -> str:
    """触发一次自动化评测（同步返回 eval_id，后台批量执行）。

    :raises RuntimeError: 已有评测在运行中（重复触发时返回 400）
    :return: eval_id（形如 eval_ab12cd34），前端立即用其轮询进度
    """
    global _running
    if _running:
        raise RuntimeError("评测已在运行中，请等待完成后再触发")
    _running = True
    eval_id = f"eval_{uuid4().hex[:8]}"
    evaluation_repository.create(eval_id, total_cases=len(TEST_CASES))
    asyncio.create_task(_execute(eval_id))  # 后台执行，接口立即返回
    logger.info("评测已启动：%s（共 %d 个用例）", eval_id, len(TEST_CASES))
    return eval_id

  runner._execute
async def _execute(eval_id: str) -> None:
    """后台批量执行全部用例 → 计算指标 → 生成 Markdown → 落库（终态）。

    任何环节异常都落到 evaluation_reports.status=failed，
    绝不静默丢失（前端可提示重试）。
    """
    global _running
    try:
        results = []
        for index, case in enumerate(TEST_CASES, start=1):
            result = await _run_single_case(case, index)
            results.append(result)
            evaluation_repository.set_progress(eval

---
## Part 6：评测落库 —— 一条评测记录的“一生”
---

### 6.1 `evaluation_reports` 表：评测记录不是“临时文件”

PRD 早期规划评测报告输出为“独立 JSON 文件目录”（`evaluation_report_{时间戳}.json`）。
第 27 节最终把评测记录**收编进 SQLite**（`data.db` 的 `evaluation_reports` 表）：
不用管文件名 / 目录 / 清理，天然支持**分页查询、删除、历史对比** —— 这和
第 24 节把任务从 `tasks.jsonl` 收编进 `tasks` 表是同一个演进逻辑（先文件后数据库）。

一张表管住“评测的一生”（状态机）：

```
create(eval_id, total_cases)              ──► status = running（前端立刻可轮询）
        │
        ▼  每跑完一个用例
set_progress(eval_id, completed_cases)     ──► completed_cases 递增（进度条数据源）
        │
        ▼  全部完成
finish(eval_id, results, metrics, md)      ──► status = completed（写结果/指标/Markdown）
        │
        ▼  整体层异常
fail(eval_id, error)                       ──► status = failed（可重新触发）
        │
        ▼  （进程重启兜底）
fail_running_interrupted()                 ──► 遗留 running → failed
```

表结构要点（`database.py` 的 `_SCHEMA`，与 tasks 同库同文件）：

| 列 | 说明 |
|----|------|
| `eval_id` | 主键（形如 `eval_ab12cd34`） |
| `status` | `running` / `completed` / `failed` |
| `total_cases` / `completed_cases` | 进度条的两个端点 |
| `results` / `metrics` | **JSON 文本列**（SQLite 没有嵌套类型） |
| `report_md` | Markdown 可视化报告（人类可读版） |
| `error` | 失败原因（status=failed 时填充） |

### 6.2 三个工程细节（读代码时注意）

| 细节 | 代码位置 | 为什么 |
|------|---------|--------|
| 结构化字段序列化**封装在仓储层** | `_dumps` / `_loads` | 业务层（runner / routes）永远拿 Python 对象，不感知“底层是 JSON 文本” |
| 更新字段有**白名单** `_ALLOWED_UPDATE` | `_update` | 防止调用方误传脏字段（如 `eval_id` / `created_at`），与 `task_repository` 同一套路 |
| `_loads` 解析失败返回默认值 | `_row_to_eval` | 老数据 / 脏数据不会让详情接口崩溃（`results` 默认 `[]`，`metrics` 默认 `{}`） |

Cell 9 走读仓储源码；Cell 10 在**临时目录**里把一条评测的一生真实走一遍：

In [9]:
# ============================================================
# Cell 9：走读 db/evaluation_repository.py —— CRUD 与状态机
# ============================================================
import inspect
from db import evaluation_repository as repo

for fn_name in ("create", "_update", "finish", "fail_running_interrupted", "list_reports"):
    print("=" * 55)
    print(f"  evaluation_repository.{fn_name}")
    print("=" * 55)
    print(inspect.getsource(getattr(repo, fn_name)))


  evaluation_repository.create
def create(eval_id: str, total_cases: int) -> None:
    """插入一条评测初始记录（status 默认 running）。

    由评测 runner 在开始前调用，保证前端提交后立刻能查到
    该评测并轮询进度（completed_cases / total_cases）。
    """
    with get_connection() as conn:
        conn.execute(
            "INSERT INTO evaluation_reports (eval_id, total_cases) VALUES (?, ?)",
            (eval_id, total_cases),
        )

  evaluation_repository._update
def _update(eval_id: str, **fields) -> None:
    """更新评测字段（结构化字段自动 JSON 序列化，并刷新 updated_at）。"""
    updates = {k: v for k, v in fields.items() if k in _ALLOWED_UPDATE}
    if not updates:
        return
    sets = []
    params: list = []
    for key, value in updates.items():
        sets.append(f"{key} = ?")
        params.append(_dumps(value) if key in _JSON_FIELDS else value)
    sets.append("updated_at = datetime('now','localtime')")
    params.append(eval_id)
    with get_connection() as conn:
        conn.execute(f"UPDATE evaluation_reports SET {', '.join(se

In [10]:
# ============================================================
# Cell 10：真实 SQLite 演示 —— 一条评测的一生（临时目录，不污染本机）
# ============================================================
import tempfile
from pathlib import Path

from db.database import init_db
from db import database
from db import evaluation_repository as repo

# 把 SQLite 指向临时文件
tmpdb = Path(tempfile.mkdtemp(prefix="sb27_life_")) / "data.db"
database.DB_PATH = tmpdb
init_db(tmpdb)
print("  临时业务库:", tmpdb)
print()

# 1) create：评测开跑
repo.create("eval_life01", total_cases=10)
rec = repo.get("eval_life01")
print("[create ] status =", rec["status"], "| total_cases =", rec["total_cases"],
      "| completed_cases =", rec["completed_cases"], "| results =", rec["results"])

# 2) set_progress：跑完 1 个用例
repo.set_progress("eval_life01", completed_cases=1)
rec = repo.get("eval_life01")
print("[progress] completed_cases =", rec["completed_cases"], "（前端进度条 1/10）")

# 3) finish：全部完成（结构化字段存取，业务层无感知 JSON 序列化）
results = [
    {"case_index": 1, "topic": "便携式储能电源 2026 北美市场", "success": True,
     "duration_seconds": 66.0, "react_loops": 6, "keyword_coverage": 1.0, "memory_hits": 1},
]
from evaluation import metrics
data = metrics.compute_metrics(results)
repo.finish("eval_life01", results, data, metrics.build_markdown("eval_life01", results, data))
rec = repo.get("eval_life01")
print("[finish ] status =", rec["status"])
print("          results 类型:", type(rec["results"]).__name__,
      "| metrics[summary][success_rate] =", rec["metrics"]["summary"]["success_rate"])
print("          report_md 第 4 行：", rec["report_md"].splitlines()[3][:60])

# 4) list_reports：列表项不带大文本（详情单查）
items, total = repo.list_reports(limit=10, offset=0)
print("[list   ] total =", total, "| 列表项字段 =", sorted(items[0].keys()))
print("          不含 results / report_md:", "results" not in items[0])

# 5) 模拟进程重启兜底：遗留 running 评测 → failed
repo.create("eval_stale01", total_cases=10)          # 假装它跑到一半进程没了
fixed = repo.fail_running_interrupted()
print("[重启兜底] 被归位的 running 评测数 =", fixed,
      "| 状态 =", repo.get("eval_stale01")["status"],
      "| error =", repo.get("eval_stale01")["error"])

# 6) delete：删除历史评测（无子表，整行移除）
print("[delete ] 删除 eval_life01:", repo.delete("eval_life01"),
      "| 再删返回:", repo.delete("eval_life01"), "（不存在 → False → API 404）")
print("          删除后查询:", repo.get("eval_life01"))


  临时业务库: C:\Users\13691\AppData\Local\Temp\sb27_life_5_w10g_y\data.db

[create ] status = running | total_cases = 10 | completed_cases = 0 | results = []
[progress] completed_cases = 1 （前端进度条 1/10）
[finish ] status = completed
          results 类型: list | metrics[summary][success_rate] = 1.0
          report_md 第 4 行： - 生成时间：2026-09-03 13:00:22
[list   ] total = 1 | 列表项字段 = ['completed_cases', 'created_at', 'error', 'eval_id', 'status', 'total_cases', 'updated_at']
          不含 results / report_md: True
[重启兜底] 被归位的 running 评测数 = 1 | 状态 = failed | error = 服务中断：评测未完成，已标记失败，可重新触发
[delete ] 删除 eval_life01: True | 再删返回: False （不存在 → False → API 404）
          删除后查询: None


---
## Part 7：后端 API —— 评测与报告怎么“开口说话”
---

### 7.1 路由层为什么这么“薄”？

打开 `api/routes.py` 你会看到一个贯穿始终的模式（从第 22 节一路用到第 27 节）：

```
① Pydantic 模型校验参数   →  ② 把“脏活”委托给业务模块   →  ③ 把业务异常翻译成 HTTP 状态码
```

每个路由函数都只有**三层**，绝不在路由里写业务逻辑。第 27 节新增的评测 API 也一样：

| API | 说明 | 关键教学点 |
|-----|------|-----------|
| `POST /api/v1/evaluation/run` | 触发一次评测（立即返回 eval_id） | `RuntimeError`（并发保护）→ **400** |
| `GET /api/v1/evaluation/reports` | 历史评测列表（分页，最新在前） | 直接透传 `list_reports()` 的 `(items, total)` |
| `GET /api/v1/evaluation/reports/{eval_id}` | 评测详情 | 执行中返回**进度**，结束返回**结果/指标/Markdown** —— 同一个接口两种形态 |
| `DELETE /api/v1/evaluation/reports/{eval_id}` | 删除历史评测 | `running` 状态返回 **400**（禁止删） |

> 为什么“业务层抛异常、路由层翻译状态码”而不是在业务层直接 `raise HTTPException`？
> 因为 **runner / repository 是纯后端逻辑，不该认识 HTTP**。
> 把它们写死在 FastAPI 上，将来换成 CLI / 定时任务调用评测就得改业务代码；
> 现在评测引擎既能被 API 触发，也能被 `python -c` 直接触发（Part 9 就这么干）。

Cell 11 直接读取 routes.py 中「第 27 节新增」的整段源码走读：

In [11]:
# ============================================================
# Cell 11：走读 api/routes.py —— 评测 4 个 API（读文件切片，不 import 整个后端）
# ============================================================
from pathlib import Path

src = Path(BACKEND) / "api" / "routes.py"
lines = src.read_text(encoding="utf-8").splitlines()

start = next(i for i, l in enumerate(lines)
             if l.startswith("# ----") and "自动化评测（第 27 节）" in l)
end = next(i for i, l in enumerate(lines)
           if l.startswith("# ----") and "任务报告（第 27 节最终完善）" in l)

print("\n".join(lines[start:end]).rstrip())


# ---------------------------------------------------------------- 自动化评测（第 27 节）


@router.post("/evaluation/run")
async def trigger_evaluation():
    """触发自动化评测：后台批量执行 10 个标准用例（约 5-10 分钟），立即返回 eval_id。

    教学要点：返回后前端用 eval_id 轮询 `GET /evaluation/reports/{eval_id}`，
    该接口在评测执行中即可返回进度（completed_cases / total_cases），
    完成后返回五维指标（JSON）与 Markdown 可视化总结。
    已有评测在运行时返回 400（并发保护），避免重复触发互相干扰。
    """
    try:
        eval_id = eval_runner.start_evaluation()
    except RuntimeError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    return {"eval_id": eval_id, "status": "running", "total_cases": eval_runner.case_count()}


@router.get("/evaluation/reports")
async def list_evaluation_reports(
    page: int = Query(1, ge=1, description="页码（从 1 开始）"),
    page_size: int = Query(10, ge=1, le=100, description="每页条数"),
):
    """历史评测报告列表（评测中心「历史评测」表格数据源，按时间倒序）。"""
    items, total = evaluation_repository.list_reports(
        limit=page_size, offset=(page - 1) * p

### 7.2 三个值得停下来想一想的 API 细节

| 细节 | 代码位置 | 为什么 |
|------|---------|--------|
| 触发接口**同步返回**，真正执行交给 `asyncio.create_task` 后台跑 | `trigger_evaluation` | 与 `POST /generate` 同一套路：长任务永远“先给 id、再轮询”，接口绝不傻等 5-10 分钟 |
| `list_evaluation_reports` 只返回摘要字段（无 `results` / `report_md`） | `list_reports()` 的 SELECT 列 | 大文本单查（详情接口），列表接口只回列表需要的东西 —— 前端历史表才不卡 |
| 删除运行中的评测会 400 | `delete_evaluation_report` | 后台 `_execute` 还在写这条记录，删了它 → 更新落 0 行、前端轮询 404，状态会“人格分裂” |

### 7.3 报告类 API：给第 26 节的“报告”装上出口

产品经理的问题二（“报告有没有中心？”）不只是加页面 —— 后端要给前端两样东西：

1. **`GET /api/v1/tasks/{task_id}/report`（新增）**：单份报告的展示数据源。
   返回 Markdown 正文 + `executive_summary` 执行摘要（总耗时 / ReAct 次数 / 记忆命中），
   注意里面的 `charts: []` —— 项目在 PRD 里**诚实标注**“Analyst 图表 Agent 属后续扩展”，
   接口先占位，将来有图了前端不用改契约。
2. **`GET /api/v1/tasks` 扩展 `keyword` + `order`**（完善）：报告中心的搜索 / 排序能力。
   第 24 节这个接口只能分页 + 按状态过滤；第 27 节给 `task_repository.list_tasks` 加了
   `keyword`（主题模糊搜索，SQL `LIKE`）与 `order`（时间升 / 降序）两个维度。

> 设计取向：**能“完善旧接口”就不“新造接口”** —— 报告中心的数据源本质还是 tasks 表，
> 只是多两个查询维度；前端 `tasks.ts` 把参数透传即可，后端逻辑零重复。

Cell 12 走读这两个报告类接口的源码：

In [12]:
# ============================================================
# Cell 12：走读 api/routes.py —— 报告中心相关接口（list_tasks 扩展 + task_report）
# ============================================================
from pathlib import Path

lines = Path(BACKEND).joinpath("api", "routes.py").read_text(encoding="utf-8").splitlines()


def show_route(name: str):
    """按函数名定位路由源码：回溯装饰器，打印到下一个顶层装饰器/定义前。"""
    i = next(i for i, l in enumerate(lines)
             if l.startswith(f"def {name}(") or l.startswith(f"async def {name}("))
    j = i
    while j > 0 and lines[j - 1].strip() == "":        # 跳过函数前的空行
        j -= 1
    if j > 0 and lines[j - 1].startswith("@router"):   # 再向上找回装饰器
        j -= 1
    end = next((k for k in range(i + 1, len(lines))
                if lines[k].startswith("@router") or lines[k].startswith("def ")
                or lines[k].startswith("# ----")), len(lines))
    print("\n".join(lines[j:end]).rstrip())


for name in ("list_tasks", "task_report", "_elapsed_seconds"):
    print("=" * 60)
    print(f"  routes.{name}")
    print("=" * 60)
    show_route(name)
    print()


  routes.list_tasks
@router.get("/tasks")
async def list_tasks(
    page: int = Query(1, ge=1, description="页码（从 1 开始）"),
    page_size: int = Query(10, ge=1, le=100, description="每页条数"),
    status: str | None = Query(default=None, description="按状态过滤：running / paused / completed / failed"),
    keyword: str | None = Query(default=None, max_length=200, description="按主题模糊搜索（报告中心搜索框）"),
    order: str = Query("desc", pattern="^(asc|desc)$", description="时间排序：desc 最新在前 / asc 最久在前"),
):
    """任务列表（分页 + 状态过滤 + 主题搜索 + 时间排序），第 27 节「报告中心」数据源。"""
    items, total = task_repository.list_tasks(
        status=status,
        keyword=keyword,
        order=order,
        limit=page_size,
        offset=(page - 1) * page_size,
    )
    return {"total": total, "page": page, "page_size": page_size, "items": items}

  routes.task_report
@router.get("/tasks/{task_id}/report")
async def task_report(task_id: str):
    """获取任务最终报告（Markdown）与执行摘要（「报告展示」页数据源）。

    教学要点：报告 = 任务成功后的 final_report（草稿即最终报告），


> 💡 `_elapsed_seconds` 解析失败返回 `None` 而不是抛异常 —— 前端对 `None` 显示“—”即可，
> 报告页不至于因为一条脏时间数据整页报错。**对外接口的“容错下限”永远放在最容易崩的那一层**。

---
## Part 8：前端闭环 —— 让“数字”能看见、能操作
---

### 8.1 前端新增了什么？

| 文件 | 角色 | 一句话职责 |
|------|------|-----------|
| `src/api/evaluation.ts` | ★ API 封装层 | 类型定义 + 4 个请求函数（运行 / 列表 / 详情 / 删除） |
| `src/views/Evaluation.vue` | ★ 评测中心 | “运行评测”按钮 + 进度条 + 指标卡 + **五维雷达图** + 用例明细 + Markdown 报告 |
| `src/views/Reports.vue` | ★ 报告中心 | 全部已完成任务的报告列表：按主题**搜索** / 按时间**排序** / 删除 |
| `src/views/ReportView.vue` | ★ 报告展示 | 渲染 `GET /tasks/{id}/report` 返回的 Markdown + 执行摘要 + 导出 |

> 命名上有个容易混的点：**评测中心（Evaluation）管“考试”**，数据来自 `evaluation_reports` 表；
> **报告中心（Reports）管“考完的卷子”**，数据来自 `tasks` 表（第 24-26 节生成的任务报告）。
> 两者在界面上都要“看报告”，但一个看批量评测的汇总，一个按主题翻历史成品。

### 8.2 API 封装层：为什么前端先写类型、再写函数？

打开 `evaluation.ts` 你会发现它大半篇幅是 `interface` —— 这不是废话，而是**“类型即契约”**：

```
后端 Pydantic / 字典的字段名  ⇄  TS interface  ⇄  Vue 组件读取
```

- TS 类型与后端返回结构**逐字段对应**（`status: 'running'|'completed'|'failed'` 字面量联合类型，
  直接把后端的合法取值写死在类型里 —— 拼错字段名或漏判一个状态，编译期就报错）；
- 前端组件只认 `EvalReport`，**不知道也不关心** axios 的请求细节；
- 加一个字段 = 后端加 + 类型加 + 页面加，三处各改各的，互不越界。

Cell 13 把整个 `evaluation.ts` 读出来走读：

In [13]:
# ============================================================
# Cell 13：走读 frontend/src/api/evaluation.ts —— 类型即契约
# ============================================================
from pathlib import Path

ts = Path(FRONTEND) / "src" / "api" / "evaluation.ts"
print(f"文件：{ts.relative_to(Path(BACKEND).parent)}（共 {len(ts.read_text(encoding='utf-8').splitlines())} 行）")
print("=" * 60)
print(ts.read_text(encoding="utf-8").rstrip())


文件：frontend\src\api\evaluation.ts（共 103 行）
import axios from 'axios'

/** 评测中心 API 封装（第 27 节） */

/** 单个用例的执行结果 */
export interface EvalCaseResult {
  case_index: number
  topic: string
  keywords: string[]
  task_id: string
  success: boolean
  /** 耗时（秒）；失败用例为 null */
  duration_seconds: number | null
  /** ReAct 循环次数 = Action 日志条数 */
  react_loops: number
  /** 关键词覆盖率（0~1，命中数/总关键词数） */
  keyword_coverage: number
  /** 长期记忆命中条数 */
  memory_hits: number
  /** 是否因达循环上限被强制通过 */
  forced_pass: boolean
  error: string | null
}

/** 评测指标汇总（实测值 + 目标值 + 达标判定） */
export interface EvalMetrics {
  summary: {
    total_cases: number
    success_cases: number
    failed_cases: number
    success_rate: number
    avg_duration_seconds: number
    p95_duration_seconds: number
    avg_react_loops: number
    keyword_coverage_rate: number
    memory_hit_rate: number
  }
  targets: Record<string, number>
  met: Record<string, boolean>
}

/** 评测报告（详情接口返回） */
export interface EvalReport {
  eval_id: strin

### 8.3 评测中心最“反直觉”的一段：雷达图的归一化

看 `Evaluation.vue` 的 `radarOption` 前，先想一个问题：

> 雷达图有 5 个维度，但其中两个（耗时 / ReAct 循环）是**越低越好**，
> 另外三个（成功率 / 覆盖率 / 记忆命中率）是**越高越好** ——
> ECharts 雷达图假设所有轴都是“越大越向外”，怎么把两类指标画进同一张图？

答案是把“越低越好”的指标**倒过来**：`得分 = 1 - 实测值 / 目标值`。
以平均耗时为例（目标 120s）：

```
实测 60s  →  (1 - 60/120) × 100 = 50 分   ← 只达到一半时间预算，得 50
实测 120s →  (1 - 120/120) × 100 =  0 分   ← 刚好卡线，0 分（贴在最内圈）
实测 0s   →  (1 - 0/120)   × 100 = 100 分   ← 秒回，满分
```

再用 `Math.max(0, Math.min(100, x))` 把结果**夹在 0~100**（防御：实测超了目标
也不允许出现负分把图撑破）。5 个维度因此统一成“**越靠外越好**”，雷达图才有意义。

这个“先统一方向、再夹紧范围”的思路，是这类多量纲可视化最常见的工程处理。
Cell 14 直接从 `Evaluation.vue` 里把指标卡 + 雷达图那一段读出来走读：

In [14]:
# ============================================================
# Cell 14：走读 Evaluation.vue —— 指标卡 + 雷达图归一化（读文件切片）
# ============================================================
from pathlib import Path

vue_lines = (Path(FRONTEND) / "src" / "views" / "Evaluation.vue") \
    .read_text(encoding="utf-8").splitlines()

start = next(i for i, l in enumerate(vue_lines) if "派生数据" in l)
end = next(i for i, l in enumerate(vue_lines) if l.startswith("const showMd"))
print("切片范围：Evaluation.vue 第 %d ~ %d 行（派生数据 → 指标卡 → 雷达图 → 图表挂载）"
      % (start + 1, end))
print("=" * 60)
print("\n".join(vue_lines[start:end]))


切片范围：Evaluation.vue 第 139 ~ 221 行（派生数据 → 指标卡 → 雷达图 → 图表挂载）
// ---------------- 派生数据 ----------------

const summary = computed(() => report.value?.metrics?.summary)

/** 指标展示卡（实测值 + 目标值 + 达标标签） */
const metricCards = computed(() => {
  const s = summary.value
  if (!s) return []
  const pct = (v: number) => `${(v * 100).toFixed(1)}%`
  const target = report.value!.metrics.targets
  const met = report.value!.metrics.met
  return [
    { key: 'success_rate', label: '任务成功率', value: pct(s.success_rate), target: pct(target.success_rate), met: met.success_rate },
    { key: 'avg_duration_seconds', label: '平均执行耗时', value: `${s.avg_duration_seconds.toFixed(1)}s`, target: `${target.avg_duration_seconds.toFixed(1)}s`, met: met.avg_duration_seconds },
    { key: 'p95_duration_seconds', label: 'P95 耗时', value: `${s.p95_duration_seconds.toFixed(1)}s`, target: '—', met: true },
    { key: 'avg_react_loops', label: '平均 ReAct 循环', value: `${s.avg_react_loops.toFixed(1)} 次`, target: `${target.avg_react

### 8.4 三个页面怎么“协作”？（不用读完全部代码，知道分工即可）

```
评测中心 Evaluation.vue（路由 /evaluation）
   ├─ 运行评测      → POST  /evaluation/run              （拿 eval_id）
   ├─ 进度轮询      → GET   /evaluation/reports/{id}     （2s 一次；见 startPolling）
   ├─ 历史列表      → GET   /evaluation/reports?page=    （点行即选中）
   ├─ 删除历史      → DELETE /evaluation/reports/{id}    （running 时按钮禁用）
   └─ 结果展示      → 指标卡(实测/目标/达标) + 雷达图(归一化) + report_md 渲染 + 用例明细表

报告中心 Reports.vue（路由 /reports）
   ├─ 列表          → GET   /tasks?status=completed&page=
   ├─ 搜索框        → GET   /tasks?keyword=xxx           （后端 LIKE 模糊匹配）
   └─ 时间排序      → GET   /tasks?order=asc|desc
   └─ 点「查看」     → 跳转 /report/{taskId}

报告展示 ReportView.vue（路由 /report/:taskId）
   ├─ 取数          → GET   /tasks/{id}/report           （Markdown + 执行摘要）
   ├─ 渲染          → marked 解析 + DOMPurify 消毒（防 XSS，安全渲染不可信 HTML）
   └─ 导出          → 打印 / 导出（html2pdf.js 类库）
```

> 注意 Reports.vue 复用的是**第 24 节的 `fetchTasks`**，只是多了 `keyword` / `order` 两个参数 ——
> 前端同样遵守“**能扩参数就不新写函数**”，与 Part 7 后端“完善旧接口”的思路一一对应。

---
## Part 9：端到端演示 —— 把整条链路“拼起来跑一遍”
---

### 9.1 演示怎么做到“离线可跑、又不失真”？

真实评测要调 LLM（10 个用例约 5-10 分钟、需要 API Key）。为了让教程可离线复现，
Part 9 用**打桩**（stub）跑通**除了 LLM 以外的一切**：

```
真实系统：  runner ──调──► workflow.run_task ──调──► LLM(5-10 分钟)
演示打桩：  runner ──调──► workflow.run_task ──桩──► sleep(0.2s) + 写 task_repository
```

关键点：**桩只替换“写报告的人”，不替换“记账的人”** ——

| 环节 | 演示里是真代码吗？ | 说明 |
|------|:---:|------|
| `start_evaluation` 并发锁 / 立即返回 | ✅ 真 | runner.py 原封不动 |
| `_execute` 逐用例执行 / 进度落库 | ✅ 真 | 每跑完一个用例写一次 `completed_cases` |
| `_run_single_case` 计时 / 采样 | ✅ 真 | 用 `time.monotonic()` 真计时（桩耗时约 0.2s/用例） |
| `task_repository` 写报告 / 日志 / 记忆 | ✅ 真 | 桩把“报告正文 + Action 日志 + memory_hits”写进真实 SQLite |
| `metrics.compute_metrics` 指标 / 达标 | ✅ 真 | 关键词覆盖率是按桩写的正文**真实子串匹配**算出来的 |
| LLM | 🧪 桩 | 0.2 秒“秒答”，只为演示链路 |

> 这正好呼应测试文件 `test_evaluation.py` 的打法 —— 用 `patch` 换掉 `workflow.run_task`，
> 其余全部走真代码。**教学版 = 把测试“放慢、打印出来给你看”**。

### 9.2 演示将依次验证四件事

| 场景 | 验证什么 | 对应测试用例 |
|------|---------|-------------|
| ① happy path：3 个用例全成功 | 全绿指标 + 进度条逐条前进 | `test_start_evaluation_and_execute` |
| ② 单用例失败 | “用例可以挂，考试不能崩” → 成功率/覆盖率亮红灯，整轮仍 completed | `test_run_single_case_failure_records_error` |
| ③ 重复触发 | 并发锁 `_running` → `RuntimeError`（API 层即 400） | `test_start_evaluation_rejects_concurrent_run` |
| ④ 进程重启兜底 | 遗留 `running` 评测被 `fail_running_interrupted` 归位 | `test_fail_running_interrupted` |

> ⚠️ Notebook 的内核本身就运行着一个 asyncio 事件循环，直接 `asyncio.run()` 会报
> “cannot be called from a running event loop”。演示用一个**独立线程 + 独立事件循环**
> 跑 async 场景 —— 这正好模拟了真实系统里 uvicorn 那个常驻事件循环的角色。

Cell 15 先搭好“三件道具”（临时库 / 演示用例集 / 桩 workflow），再跑场景①和②：

In [15]:
# ============================================================
# Cell 15：场景①② —— happy path 全绿 + 单用例失败不崩
# ============================================================
import asyncio
import logging
import tempfile
import threading
from pathlib import Path

from db import database
from db import evaluation_repository as repo
from db import task_repository
from evaluation import runner

# 屏蔽 runner 内部 logger.exception（单用例失败时它会打印 traceback，刷屏且误导）
logging.getLogger("evaluation.runner").disabled = True

# ---- 道具 1：SQLite 指向临时目录（不污染本机 smartbrief/backend/data.db） ----
tmpdb = Path(tempfile.mkdtemp(prefix="sb27_e2e_")) / "data.db"
database.DB_PATH = tmpdb
database.init_db(tmpdb)
print("演示业务库（临时）:", tmpdb)

# ---- 道具 2：恢复 3 个真实主题（runner.py 里被注释掉的那批）----
#      目的：让“状态机”有 1/3 → 2/3 → 3/3 的进度可观察；想跑满 10 个去掉注释即可
runner.TEST_CASES = [
    {"topic": "家用储能系统 2026 欧洲市场", "keywords": ["市场规模", "政策", "头部玩家", "发展趋势", "家用储能"]},
    {"topic": "智能手表 2026 中国市场分析", "keywords": ["市场规模", "竞品", "用户需求", "发展趋势", "智能手表"]},
    {"topic": "户用屋顶光伏 2026 美国市场", "keywords": ["市场规模", "政策", "装机量", "头部玩家", "光伏"]},
]

STOP_TOPICS = set()          # 场景② 往这里塞一个主题 = 让它“模拟失败”

# ---- 道具 3：桩 workflow.run_task（不调 LLM，但写真实 task_repository）----
async def fake_workflow(task_id: str, topic: str, resume: bool = False):
    await asyncio.sleep(0.2)                       # 压缩版“执行耗时”（真实为分钟级）
    if topic in STOP_TOPICS:                       # 需要演示失败的主题直接抛异常
        raise RuntimeError("模拟 LLM 调用超时（教学故意失败）")
    keywords = next(c["keywords"] for c in runner.TEST_CASES if c["topic"] == topic)
    hit = keywords if "智能手表" in topic else keywords[:4]   # 手表主题全覆盖，其余 4/5
    report_text = "本报告要点：" + "、".join(hit) + "。" + "（正文内容略）" * 30
    memory_hits = [task_id] if ("家用储能" in topic or "智能手表" in topic) else []
    task_repository.update_task(task_id, status="completed",
                                final_report=report_text, memory_hits=memory_hits)
    loops = {"家用储能系统 2026 欧洲市场": 4, "智能手表 2026 中国市场分析": 5,
             "户用屋顶光伏 2026 美国市场": 6}[topic]
    for i in range(loops):                         # Action 日志 = ReAct 循环计数来源
        task_repository.append_log(task_id, 1, "Action", f"web_search 第 {i + 1} 轮")

runner.workflow.run_task = fake_workflow


def run_async(coro_factory):
    """把 async 评测流程放进“独立线程 + 独立事件循环”运行。
    教学点：真实系统里这个常驻循环就是 FastAPI/uvicorn 的事件循环。"""
    box = {}

    def _worker():
        loop = asyncio.new_event_loop()
        try:
            box["ret"] = loop.run_until_complete(coro_factory())
        finally:
            loop.close()

    t = threading.Thread(target=_worker)
    t.start()
    t.join()
    return box.get("ret")


# ---------------- 场景①：happy path（全绿） ----------------
async def scenario_happy():
    eval_id = runner.start_evaluation()            # 同步返回，后台立刻开跑
    print(f"[start ] start_evaluation() → {eval_id}（立即可轮询）")
    last = -1
    while True:
        rec = repo.get(eval_id)
        if rec["completed_cases"] != last:         # 只在进度变化时打印，模拟前端轮询
            last = rec["completed_cases"]
            print(f"[poll  ] 进度 {last}/{rec['total_cases']}  status={rec['status']}")
        if rec["status"] != "running":
            return rec
        await asyncio.sleep(0.05)


print("\n" + "=" * 62)
print("  场景① happy path：3 个用例全部成功（观察：进度逐条前进）")
print("=" * 62)
rec = run_async(scenario_happy)
s = rec["metrics"]["summary"]
m = rec["metrics"]["met"]
print(f"[done  ] status = {rec['status']}  |  success = {s['success_cases']}/{s['total_cases']}")
print("  五维指标实测值 vs 达标：")
order = [("success_rate", "成功率"), ("avg_duration_seconds", "平均耗时"),
         ("avg_react_loops", "ReAct循环"), ("keyword_coverage_rate", "关键词覆盖"),
         ("memory_hit_rate", "记忆命中")]
for key, label in order:
    v = s[key]
    print(f"    {label:<6} = {v:<8}  {'✅ 达标' if m[key] else '❌ 未达标'}")
print("  （耗时≈0.2s/用例是桩的 sleep；真实运行是 5-10 分钟的分钟级）")

# ---------------- 场景②：单用例失败 ----------------
async def scenario_fail():
    STOP_TOPICS.add("户用屋顶光伏 2026 美国市场")   # 让第 3 个用例模拟 LLM 超时
    eval_id = runner.start_evaluation()
    while repo.get(eval_id)["status"] == "running":
        await asyncio.sleep(0.05)
    return repo.get(eval_id)


print("\n" + "=" * 62)
print("  场景② 单用例失败：用例可以挂，考试不能崩")
print("=" * 62)
rec = run_async(scenario_fail)
print("[done  ] status = %s（整轮仍正常收尾，没有烂尾）" % rec["status"])
for r in rec["results"]:
    print(f"    #{r['case_index']} {'✅' if r['success'] else '❌'} {r['topic'][:14]:<16} "
          f"error={r['error']}")
s = rec["metrics"]["summary"]
m = rec["metrics"]["met"]
for key, label in order:
    v = s[key]
    print(f"    {label:<6} = {v:<8}  {'✅ 达标' if m[key] else '❌ 未达标'}")
print("  ↑ 成功率 2/3≈0.67 < 0.9、覆盖率 (0.8+1.0+0)/3≈0.6 < 0.7 亮红灯 —— 失败被如实记录")
STOP_TOPICS.clear()


演示业务库（临时）: C:\Users\13691\AppData\Local\Temp\sb27_e2e_ul88ewp0\data.db

  场景① happy path：3 个用例全部成功（观察：进度逐条前进）
[start ] start_evaluation() → eval_fb43f442（立即可轮询）
[poll  ] 进度 0/3  status=running
[poll  ] 进度 1/3  status=running
[poll  ] 进度 2/3  status=running
[poll  ] 进度 3/3  status=completed
[done  ] status = completed  |  success = 3/3
  五维指标实测值 vs 达标：
    成功率    = 1.0       ✅ 达标
    平均耗时   = 0.22      ✅ 达标
    ReAct循环 = 5.0       ✅ 达标
    关键词覆盖  = 0.87      ✅ 达标
    记忆命中   = 0.6667    ✅ 达标
  （耗时≈0.2s/用例是桩的 sleep；真实运行是 5-10 分钟的分钟级）

  场景② 单用例失败：用例可以挂，考试不能崩
[done  ] status = completed（整轮仍正常收尾，没有烂尾）
    #1 ✅ 家用储能系统 2026 欧洲   error=None
    #2 ✅ 智能手表 2026 中国市场   error=None
    #3 ❌ 户用屋顶光伏 2026 美国   error=模拟 LLM 调用超时（教学故意失败）
    成功率    = 0.6667    ❌ 未达标
    平均耗时   = 0.23      ✅ 达标
    ReAct循环 = 3.0       ✅ 达标
    关键词覆盖  = 0.6       ❌ 未达标
    记忆命中   = 0.6667    ✅ 达标
  ↑ 成功率 2/3≈0.67 < 0.9、覆盖率 (0.8+1.0+0)/3≈0.6 < 0.7 亮红灯 —— 失败被如实记录


### 9.3 场景③④：并发保护 + 进程重启兜底

前两个场景证明了“考试能顺利考完”。还差两块拼图：

- **并发保护**：评测在后台 `asyncio.create_task` 里跑，用户手滑连点两次「运行评测」怎么办？
  `_running` 全局锁会拒绝第二次 —— 这正是 `POST /evaluation/run` 返回 400 的源头。
- **中断兜底**：进程被强杀时，后台任务没机会写终态，SQLite 会残留一条 `running` 评测。
  `main.py` 启动时调用 `fail_running_interrupted()` 把它们统一归位成 `failed`（可重开），
  前端评测中心的进度条才不会被一条“幽灵记录”永久卡死。

In [16]:
# ============================================================
# Cell 16：场景③④ —— 并发锁拒绝重复触发 + 启动兜底归位幽灵记录
# ============================================================
# ---------------- 场景③：重复触发被并发锁拒绝 ----------------
async def scenario_guard():
    eval_id = runner.start_evaluation()            # 第一次：立即占用 _running 锁
    print(f"[first ] {eval_id} 开始执行（占用并发锁 _running=True）")
    try:
        runner.start_evaluation()                  # 第二次：锁还在 → RuntimeError
        print("  !! 第二次触发竟然成功了（不应该）")
    except RuntimeError as exc:
        print("[guard ] 第二次触发被拒 →", repr(exc))
        print("         真实系统里 routes.py 捕获它并返回 HTTP 400")
    while repo.get(eval_id)["status"] == "running":
        await asyncio.sleep(0.05)
    print(f"[after ] 第一次评测 status={repo.get(eval_id)['status']}，"
          f"后台 finally 释放锁 → _running={runner._running}")


print("=" * 62)
print("  场景③ 并发保护：同一时刻只允许一场评测")
print("=" * 62)
run_async(scenario_guard)

# ---------------- 场景④：进程重启兜底（main.py lifespan 同款逻辑） ----------------
print("\n" + "=" * 62)
print("  场景④ 中断兜底：遗留 running 评测 → 统一置 failed（可重新触发）")
print("=" * 62)
repo.create("eval_ghost_01", total_cases=10)       # 假装上次进程跑到一半被强杀
repo.create("eval_ghost_02", total_cases=10)
print("[create] 库里留下 2 条永远 running 的“幽灵评测”（真实场景：进程被 kill 后残留）")

fixed = repo.fail_running_interrupted()
print(f"[fix   ] fail_running_interrupted() 归位 {fixed} 条 → "
      f"ghost_01={repo.get('eval_ghost_01')['status']}, "
      f"ghost_02={repo.get('eval_ghost_02')['status']}")
print(f"[fix   ] 第二次调用返回 {repo.fail_running_interrupted()}（幂等：没有 running 就不动）")
print("         演示里手动调用；真实系统在 main.py 的 lifespan 启动时自动执行")

# ---------------- 收尾：评测记录的“终态”一览 ----------------
print("\n" + "=" * 62)
print("  当前演示库里的评测记录（终态一览）")
print("=" * 62)
items, total = repo.list_reports(limit=20, offset=0)
for it in items:
    print(f"    {it['eval_id']:<14} status={it['status']:<10} "
          f"{it['completed_cases']}/{it['total_cases']}  {it['created_at']}")
print(f"  （共 {total} 条 —— 列表项不带 results/report_md 大文本，详情单独查）")


  场景③ 并发保护：同一时刻只允许一场评测
[first ] eval_5682b554 开始执行（占用并发锁 _running=True）
[guard ] 第二次触发被拒 → RuntimeError('评测已在运行中，请等待完成后再触发')
         真实系统里 routes.py 捕获它并返回 HTTP 400
[after ] 第一次评测 status=completed，后台 finally 释放锁 → _running=False

  场景④ 中断兜底：遗留 running 评测 → 统一置 failed（可重新触发）
[create] 库里留下 2 条永远 running 的“幽灵评测”（真实场景：进程被 kill 后残留）
[fix   ] fail_running_interrupted() 归位 2 条 → ghost_01=failed, ghost_02=failed
[fix   ] 第二次调用返回 0（幂等：没有 running 就不动）
         演示里手动调用；真实系统在 main.py 的 lifespan 启动时自动执行

  当前演示库里的评测记录（终态一览）
    eval_ghost_02  status=failed     0/10  2026-09-03 13:00:25
    eval_ghost_01  status=failed     0/10  2026-09-03 13:00:25
    eval_5682b554  status=completed  3/3  2026-09-03 13:00:24
    eval_fb43f442  status=completed  3/3  2026-09-03 13:00:23
    eval_ccc6d9e2  status=completed  3/3  2026-09-03 13:00:23
  （共 5 条 —— 列表项不带 results/report_md 大文本，详情单独查）


### 9.4 这四个场景证明了什么？

| 观察到的现象 | 背后的工程保证 | 用户视角 |
|-------------|---------------|---------|
| 进度 1/3 → 2/3 → 3/3 逐条出现 | `_execute` 每完成一用例就 `set_progress` 落库 | 前端进度条不会“假死” |
| 单用例失败整轮仍 completed | `_run_single_case` 的 `except` 兜底 | 一场考试不会因一道题崩掉 |
| 成功率 / 覆盖率变红 | `metrics.compute_metrics` 的 `met` 判定 | 指标红绿灯真实反映质量 |
| 第二次触发被 400 | runner 模块级 `_running` 并发锁 | 不会同时烧两场 LLM 的钱 |
| 重启后没有“永远 running” | lifespan → `fail_running_interrupted` | 评测中心不卡、按钮可点 |

> 想跑**真实**评测？启动后端后让前端「评测中心」点一次「运行评测」即可 ——
> 它走的正是这里演示的同一条链路，只是 `workflow.run_task` 换成真 LLM。
> 你也可以在命令行直接触发：`python -c "from evaluation.runner import start_evaluation; print(start_evaluation())"`。

---
## Part 10：交付 —— 从“我电脑能跑”到“谁都能跑”
---

### 10.1 交付分三层：给三种人用

产品经理的问题三（“总不能把命令行教程发给客户”）拆开看，其实要交付给**三种人**：

| 层级 | 给谁 | 产物 | 命令 |
|------|------|------|------|
| ① 本机一键启动 | 开发者自己 / 现场演示 | `start.bat`（Windows）/ `start.sh`（Linux/Mac） | 双击 / `bash start.sh` |
| ② 容器化编排 | 有 Docker 的环境（本地验收、内网服务器） | `deploy/docker-compose.yml` | `docker compose up -d` |
| ③ 云服务器交付 | 无 Docker 构建能力的远端主机 | `deploy/docker-compose.prod.yml` + `deploy-to-server.sh` | 本机打包镜像 → scp → 远端 `docker load` + `up` |

> 一句话记忆：**①②解决“启动繁琐”，③解决“网络/构建环境不可控”**。
> ①告诉你怎么跑，②帮你隔离环境，③把“本地能跑的镜像”原样搬到云端 —— 客户端看到的永远只有一句话：`访问 http://<服务器>:8080`。

Cell 17 先把两个一键启动脚本读出来，并看看 `deploy/` 目录里到底有什么：

In [17]:
# ============================================================
# Cell 17：走读 start.bat / start.sh —— “双击即跑”的工程细节
# ============================================================
import os
from pathlib import Path

ROOT = Path(BACKEND).parent          # smartbrief/ 项目根
print("=" * 60)
print("  start.bat（Windows 一键启动）")
print("=" * 60)
print((ROOT / "start.bat").read_text(encoding="utf-8").rstrip())

print()
print("=" * 60)
print("  start.sh（Linux / Mac 一键启动）")
print("=" * 60)
print((ROOT / "start.sh").read_text(encoding="utf-8").rstrip())

print()
print("=" * 60)
print("  deploy/ 交付目录清单")
print("=" * 60)
for p in sorted((ROOT / "deploy").rglob("*")):
    if p.is_file():
        rel = p.relative_to(ROOT / "deploy")
        print("    " + str(rel))


  start.bat（Windows 一键启动）
@echo off
REM =====================================================================
REM SmartBrief one-click launcher (Windows)
REM - Backend FastAPI (port 8002): starts in a separate window.
REM   The built-in MCP Server is auto-started by the backend.
REM - Frontend Vite (port 5174): starts in the foreground, Ctrl+C to stop.
REM Prerequisites:
REM   - configure backend/.env (OPENAI_* / EMBEDDING_*)
REM   - pip install -r backend/requirements.txt
REM   - cd frontend && npm install
REM =====================================================================
cd /d "%~dp0"

echo [SmartBrief] Starting backend (http://localhost:8002) ...
start "SmartBrief-Backend" cmd /k "cd /d backend && uvicorn main:app --host 0.0.0.0 --port 8002"

echo [SmartBrief] Starting frontend (http://localhost:5174) ...
cd frontend
call npm run dev

  start.sh（Linux / Mac 一键启动）
#!/usr/bin/env bash
# =====================================================================
# SmartBrief 一键启动脚本（Li

### 10.2 一键启动脚本里藏着哪些“工程细节”？（不读脚本根本想不到）

| 脚本里的写法 | 为什么 |
|------------|--------|
| `start.bat` 用 `start "SmartBrief-Backend" cmd /k ...` **开新窗口**跑后端 | 两个长驻进程放同一窗口，Ctrl+C 会一起杀掉，看不到后端日志；新窗口各自独立 |
| `start.sh` 用 `&` 后台跑后端 + `trap 'kill %1' EXIT` | 前端窗口关闭（Ctrl+C）时自动回收后台后端进程，**不留僵尸进程** |
| 后端统一 `uvicorn main:app` 而不是再写一层启动器 | 内置 MCP Server 由后端 lifespan **自动拉起**，用户不需要知道 MCP 的存在 |
| 脚本头部注释“首次运行请先 pip install / npm install” | 脚本只负责“启动”，**依赖安装是另一个生命周期** —— 混在一起会让失败定位变难 |
| 端口写死 8002（后端）/ 5174（前端） | 与 vite.config / 前端 axios baseURL 约定一致，避免“端口漂移”导致的联调玄学 |

> 对比第 22 节你最早可能是“开两个终端手动敲 uvicorn + npm run dev” ——
> 现在把这两条命令连同约定一起**固化成文件**，就是“交付”的起点：
> 让它短到**任何人**都能跑，而不是只有你记得住。

### 10.3 容器化：为什么“镜像即交付”能治本？

本地脚本治标不治本 —— 换一台机器可能 Python 版本、依赖、系统库全不一样。
Docker 的思路是：**把“能跑的环境”打成镜像，连环境一起搬走**。项目里这套编排有几个刻意设计：

| 设计 | 对应文件 | 解决什么问题 |
|------|---------|-------------|
| 构建上下文 = 仓库根，`COPY` 只拷 backend + mcp_servers | `deploy/backend/Dockerfile` | MCP Server 脚本路径按 `/app/mcp_servers` 计算，**目录布局必须保持**（代码注释里专门提醒） |
| 先拷 `requirements.txt` 单独 `RUN pip install` | Dockerfile | **层缓存**：依赖没变就不重装，反复构建秒级完成 |
| `data/` 用 volume 挂载，seed 单独备份到 `/app/seed` 由 entrypoint 首次复制 | Dockerfile + entrypoint.sh | 空卷会“盖住”镜像内同路径内容 —— 这是容器化最容易踩的坑之一 |
| `.env` 走 `env_file` 注入，镜像内**不装任何 Key** | docker-compose | 镜像可安全分发/进仓库，密钥留在服务器本地的 `.env` |
| 后端带 `healthcheck`，前端 `depends_on: service_healthy` | compose | 后端 RAG seed 初始化约需 40s，前端等它健康再启动，避免首屏 502 |
| 生产 compose **没有 build**，只用 `image:` | `docker-compose.prod.yml` | 云服务器可能没网络拉依赖，**本地打包好镜像直接 load** |

Cell 18 走读三份“交付产物”：后端 Dockerfile、生产编排、一键上云脚本：

In [18]:
# ============================================================
# Cell 18：走读交付产物 —— Dockerfile + docker-compose.prod.yml + 上云脚本
# ============================================================
from pathlib import Path

ROOT = Path(BACKEND).parent          # smartbrief/ 项目根

files = [
    ("deploy/backend/Dockerfile（后端镜像构建：分步缓存 / seed 备份 / CRLF 清理）",
     ROOT / "deploy" / "backend" / "Dockerfile"),
    ("deploy/docker-compose.prod.yml（云端编排：无 build，镜像直接 load）",
     ROOT / "deploy" / "docker-compose.prod.yml"),
    ("deploy/deploy-to-server.sh（一键上云：build → save → scp → load → up）",
     ROOT / "deploy" / "deploy-to-server.sh"),
]
for title, path in files:
    print("=" * 60)
    print("  " + title)
    print("=" * 60)
    print(path.read_text(encoding="utf-8").rstrip())
    print()


  deploy/backend/Dockerfile（后端镜像构建：分步缓存 / seed 备份 / CRLF 清理）
# =====================================================================
# SmartBrief 后端镜像（FastAPI + 多智能体编排引擎 + MCP Server）
# 构建上下文 = 仓库根目录（smartbrief/），由 docker-compose.yml 指定
#   说明：后端在容器内会以子进程拉起 mcp_servers/web_search_server.py，
#   其脚本路径按 /app/mcp_servers 计算（见 mcp_client.py 的 parents[2]），
#   因此目录布局必须保持 WORKDIR=/app + backend/ + mcp_servers/。
# =====================================================================
FROM python:3.13-slim-bookworm

# 运行期环境：不缓冲日志（docker compose logs 实时可见）、不写 pyc、pip 不缓存
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1

WORKDIR /app

# 1) 先只拷贝依赖清单并安装（层缓存：requirements.txt 不变则不会重装）
COPY backend/requirements.txt /app/backend/requirements.txt
RUN pip install --no-cache-dir -r /app/backend/requirements.txt

# 2) 拷贝后端源码与 MCP Server（MCP 由后端以子进程拉起，两处都需要）
COPY backend/ /app/backend/
COPY mcp_servers/ /app/mcp_servers/

# 3) RAG 预置 seed（模板 + 术语）单独备份到 /app/seed：
#    data/ 目录

### 10.4 上云脚本的四步：为什么是“搬镜像”而不是“在云端构建”？

`deploy-to-server.sh` 用 4 步把服务搬到任意 SSH 主机：

```
① docker compose build        本机构建两个镜像（网络好、有缓存，构建最快）
② docker save | gzip          镜像 → 单个 tar.gz（把“环境”打包成“文件”）
③ scp tar.gz + compose + .env 经 SSH 加密通道上传（.env 含密钥，只在可信服务器用）
④ docker load + compose up -d 远端解包镜像并启动；.env 不存在则提示先配置再跑
```

> 关键取舍：**不在云端 build**。云服务器网络慢、且可能拉不动基础镜像；
> 本地构建好“一个 tar.gz”，上传后 `docker load` 秒级还原 ——
> 这正是“镜像即交付”：**交付的不是代码，是可运行的镜像**。
>
> 最后一条命令打头带 `ssh "$SERVER" " ... "` 里的注释还演示了一个小技巧：
> 服务器端首次没有 `.env` 时 `exit 1` 并提示 —— **脚本宁可失败，也不要带着空 Key 起服务**。

---
## Part 11：课程总结 —— 项目完结，系统“可衡量、可证明、可交付”
---

### 11.1 第 27 节只做了一件事：把“功能”变成“交付物”

回顾开头的“灵魂三问”，答案已经全部落进代码：

| 问题 | 交付物 | 在哪里 |
|------|--------|--------|
| ① 拿什么证明系统靠谱？ | 10 个标准用例 + 五维指标 + 达标红绿灯 | `evaluation/metrics.py` + `runner.py` |
| ② 报告有没有统一出口？ | 评测中心 / 报告中心 / 报告展示三个页面 | 后端 7 个 API + 前端 3 个 View |
| ③ 交付 ≠ 能跑起来？ | 一键启动 + Docker Compose + 云部署脚本 | `start.*` + `deploy/` |

### 11.2 六个部分的“设计→代码”回顾（本节的完整逻辑链）

```
① 定口径：   五维指标 + TARGETS + met 红绿灯（纯计算，可单测钉死）
   ↓
② 出考卷：   10 个标准 Topic + 预期关键词（主题刻意对齐模板行业）
   ↓
③ 无人监考： runner 逐用例执行、进度落库、用例失败不崩、并发锁
   ↓
④ 留档案：   evaluation_reports 表 create→progress→finish/fail（重启兜底）
   ↓
⑤ 开口说话： 评测/报告/列表搜索排序 API（业务异常 → HTTP 状态码）
   ↓
⑥ 让人看见： 前端评测中心（进度条/指标卡/雷达图）+ 报告中心 + 报告展示
   ↓
⑦ 能交付：   一键启动 + 容器镜像 + 云服务器四步部署
```

> 如果只记一句话：**第 27 节的全部代码，都在回答“如何让一个靠 LLM 吃饭的、
> 非确定性的系统，变得可验收、可留档、可交付”** —— 而这恰恰是 Agent 项目
> 从“demo”走向“产品”的分水岭。

### 11.3 放在整个课程里看（v22 → v27 的演化主线）

| 版本 | 一句话里程碑 | 第 27 节如何接住它 |
|------|-------------|-------------------|
| v22 对接大模型 + 工具调用 | 系统会“动手”了 | 评测的用例就是调它 |
| v23 技能池 | 会“组织招式” | 无需改动，评测只看产物 |
| v24 Plan-and-Execute + ReAct | 会“规划 + 执行” | `_run_single_case` 直接复用它；`avg_react_loops` 度量它的收敛 |
| v25 A2A 审核与反馈循环 | 会“自我把关” | 结果里的 `forced_pass` 让考官记录“放水” |
| v26 RAG + 记忆 | 有知识、有记忆 | `keyword_coverage_rate` / `memory_hit_rate` 验证它们真的生效 |
| v27 全栈评测 + 交付 | **可衡量、可证明、可交付** | 本项目完结 🎉 |

> 注意第 27 节**没有新增任何 LLM 能力** —— 它做的是“度量与交付”。
> 这提醒我们：**Agent 系统的工程化（评测/记忆留档/部署）和 Agent 本身一样重要**，
> 前者才是它能否被信任、被长期维护的关键。

Cell 19 做“验收自查”：看一眼第 27 节测试到底覆盖了什么、怎么一键跑：

In [19]:
# ============================================================
# Cell 19：验收自查 —— 第 27 节测试覆盖了哪些行为？
# ============================================================
import ast
from pathlib import Path

tests_dir = Path(BACKEND) / "tests"
for name in ("test_evaluation.py", "test_evaluation_api.py"):
    source = (tests_dir / name).read_text(encoding="utf-8")
    tree = ast.parse(source)
    fns = [n.name for n in ast.walk(tree)
           if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")]
    print("=" * 60)
    print(f"  {name}（{len(fns)} 个测试）")
    print("=" * 60)
    for fn in fns:
        print("    - " + fn)
    print()

print("一条命令跑完第 27 节全部测试：")
print("    cd smartbrief/backend && python -m pytest tests/test_evaluation.py "
      "tests/test_evaluation_api.py -q")
print()
print("（它们全部离线可跑：评测测试用 patch 打桩 workflow.run_task，不发真实 LLM）")


  test_evaluation.py（14 个测试）
    - test_compute_keyword_coverage
    - test_count_react_actions
    - test_compute_p95
    - test_compute_metrics_and_targets
    - test_build_markdown_contains_tables
    - test_evaluation_repository_lifecycle
    - test_fail_running_interrupted
    - test_evaluation_repository_delete
    - test_evaluation_repository_fail_and_list
    - test_run_single_case_success
    - test_run_single_case_failure_records_error
    - test_start_evaluation_and_execute
    - test_start_evaluation_rejects_concurrent_run
    - test_exception_marks_report_failed

  test_evaluation_api.py（8 个测试）
    - test_trigger_evaluation_returns_eval_id
    - test_trigger_evaluation_rejects_concurrent_run
    - test_list_and_get_evaluation_reports
    - test_delete_evaluation_report
    - test_evaluation_report_not_found
    - test_lifespan_marks_running_evals_failed
    - test_task_report_endpoint
    - test_task_report_not_completed_or_missing

一条命令跑完第 27 节全部测试：
    cd smartbrief/back

### 11.4 课后练习（动手把“最后一公里”走完）

**练习 1：跑满 10 个用例**
打开 `backend/evaluation/runner.py`，把 `TEST_CASES` 里其余 9 条注释取消，
重新启动评测 —— 观察进度条 1/10 → 10/10、以及 5-10 分钟的真实耗时。
思考：为什么第 27 节刻意默认只开 1 个用例？

**练习 2：给系统加“第六维”指标 —— LLM 打分**
当前关键词覆盖率是“子串匹配”（便宜、确定、可复现）。请讨论并在 `metrics.py` 中
试验：新增 `llm_quality_score`（请 LLM 给报告质量打分）会带来什么收益与代价？
口径变了，`TARGETS`、`compute_metrics`、前端 `metricCards`/`radarOption` 要动几处？
（提示：先在 `metrics.py` 写纯计算，再想怎么让 runner 采样到它 —— 这就是 4.1 的“两半拆分”。）

**练习 3：把历史评测画成“趋势线”**
评测中心现在只能看单次。请在前端加一个“历史对比”视图：把多次评测的
`success_rate` 按时间画折线。思考数据源该用哪个 API？要不要后端加聚合接口？

**练习 4：真·云部署**
按 `deploy/README.md` 走一遍：本地 `docker compose build` → `./deploy-to-server.sh root@<ip>`
→ 浏览器访问 `http://<ip>:8080`。若没有云服务器，也可以用一台虚拟机或 Docker Desktop 模拟。
思考：`.env`（含 API Key）为什么绝不能进镜像、进 git？

### 11.5 参考资料

| 资源 | 位置 |
|------|------|
| 第 27 节规格（PRD 子文档 6） | `多智能体协作开发PRD（拆分）.md` → 「子文档 6：第 27 节」 |
| 项目宪法（迭代记录 / 设计决策） | `smartbrief/CLAUDE.md` |
| 评测模块源码 | `backend/evaluation/metrics.py`、`runner.py` |
| 评测持久化 + 表结构 | `backend/db/evaluation_repository.py`、`db/database.py` |
| API 路由（评测 / 报告 / 列表扩展） | `backend/api/routes.py` |
| 自动化测试 | `backend/tests/test_evaluation.py`、`test_evaluation_api.py` |
| 前端评测 / 报告页 | `frontend/src/views/Evaluation.vue`、`Reports.vue`、`ReportView.vue` |
| 一键启动与部署 | `start.bat`、`start.sh`、`deploy/` |
| 前置课程 | `24-多智能体规划落地与ReAct执行.ipynb`、`25-A2A协同与反馈循环.ipynb`、`26-RAG知识库与智能体记忆.ipynb` |

---

> 🎓 **致学员**：到此，SmartBrief 从一句“帮我写储能市场报告”出发，经历了
> 工具调用、技能池、多智能体规划协同、A2A 审核、RAG 与记忆，最终在第 27 节完成了
> **“能给自己打分、能把报告归档、能一键交付”** 的全栈闭环。
> 前六节让你学会“造一个聪明的 Agent”，这一节教你的，是“如何让别人相信它、使用它、运维它” ——
> 愿你在真实项目中，也把这两件事同等看重。**完结，鼓掌 👏**